In [6]:
import os
import sys
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
from sklearn.impute import SimpleImputer
import joblib
import warnings
warnings.filterwarnings("ignore")

sys.path.append("../src")
from feature_extraction.audio_features import extract_features

In [7]:
dataset = []
ravdess_path = "../data/ravdess"

for actor_folder in os.listdir(ravdess_path):
    actor_path = os.path.join(ravdess_path, actor_folder)
    if not os.path.isdir(actor_path):
        continue
    for filename in os.listdir(actor_path):
        if not filename.endswith(".wav"):
            continue
        parts = filename.replace(".wav", "").split("-")
        if len(parts) < 3:
            continue
        emotion = int(parts[2])
        # Label: 0 = no stress, 1 = stressed
        if emotion in [1, 2, 3]:      # neutral, calm, happy
            label = 0
        elif emotion in [5, 6, 7]:    # angry, fearful, disgust
            label = 1
        else:
            continue  # skip sad and surprised
        filepath = os.path.join(actor_path, filename)
        try:
            features = extract_features(filepath)
            features["file"]  = filepath
            features["label"] = label
            dataset.append(features)
        except Exception as e:
            print(f"❌ {filename}: {e}")

df = pd.DataFrame(dataset)
print(f"Dataset shape: {df.shape}")
print(df["label"].value_counts())

Dataset shape: (1056, 21)
label
1    576
0    480
Name: count, dtype: int64


In [8]:
os.makedirs("../data/voice_features", exist_ok=True)
df.to_csv("../data/voice_features/stress_dataset.csv", index=False)
print("✅ Saved stress_dataset.csv")

✅ Saved stress_dataset.csv


In [9]:
feature_cols = [
    "pitch", "spectral_centroid", "zcr",
    "jitter", "shimmer", "hnr",
    "mfcc_1","mfcc_2","mfcc_3","mfcc_4","mfcc_5","mfcc_6","mfcc_7",
    "mfcc_8","mfcc_9","mfcc_10","mfcc_11","mfcc_12","mfcc_13"
]

X = df[feature_cols]
y = df["label"]

imputer = SimpleImputer(strategy="mean")
X_imp   = imputer.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_imp, y, test_size=0.2, random_state=42, stratify=y
)

model = RandomForestClassifier(n_estimators=200, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print(f"Accuracy : {accuracy_score(y_test, y_pred):.4f}")
print(f"AUROC    : {roc_auc_score(y_test, y_prob):.4f}")
print(classification_report(y_test, y_pred, target_names=["No Stress", "Stressed"]))

Accuracy : 0.8349
AUROC    : 0.9236
              precision    recall  f1-score   support

   No Stress       0.87      0.75      0.80        96
    Stressed       0.81      0.91      0.86       116

    accuracy                           0.83       212
   macro avg       0.84      0.83      0.83       212
weighted avg       0.84      0.83      0.83       212



In [10]:
os.makedirs("../models", exist_ok=True)
joblib.dump(model,   "../models/stress_model.pkl")
joblib.dump(imputer, "../models/stress_imputer.pkl")
print("✅ Saved stress_model.pkl")
print("✅ Saved stress_imputer.pkl")

✅ Saved stress_model.pkl
✅ Saved stress_imputer.pkl


In [11]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
model_cv = RandomForestClassifier(n_estimators=200, random_state=42)

cv_acc = cross_val_score(model_cv, X_imp, y, cv=skf, scoring="accuracy")
cv_auc = cross_val_score(model_cv, X_imp, y, cv=skf, scoring="roc_auc")

print("Stress Model - 5-Fold Cross Validation:")
print(f"  Accuracy : {cv_acc.mean():.4f} +/- {cv_acc.std():.4f}")
print(f"  AUROC    : {cv_auc.mean():.4f} +/- {cv_auc.std():.4f}")

Stress Model - 5-Fold Cross Validation:
  Accuracy : 0.8362 +/- 0.0262
  AUROC    : 0.9184 +/- 0.0173
